# Baseline на ALS-разложениях и TopPop

In [1]:
import sys

sys.path.append('/Users/n.mamashakirov/PycharmProjects/hseml-group-project-nurtilekmamashakirov-2')

### Загрузка и сплит датасета

In [36]:
from src.data.yambda_dataset import get_yambda_pandas
from src.data.utils import split_data_by_timestamp, get_ground_truth

multi_events_df = get_yambda_pandas()

TRAIN_THRESHOLD = 0.8

train_df, test_df = split_data_by_timestamp(multi_events_df, TRAIN_THRESHOLD)
ground_truth = get_ground_truth(test_df=test_df, played_ratio_threshold=80.0)
train_users = set(train_df['uid'])
ground_truth_filtered = ground_truth[ground_truth.index.isin(train_users)]

Training set size: 38232359 rows
Testing set size: 9558090 rows


In [37]:
train_df.head()

,uid,timestamp,item_id,is_organic,played_ratio_pct,track_length_seconds,event_type
0,468300,0,7400764,0,100.0,225.0,listen
1,347600,5,3415205,0,100.0,250.0,listen
2,942900,10,6728180,0,1.0,270.0,listen
3,12700,15,8932363,0,100.0,245.0,listen
4,243500,15,5283544,1,100.0,195.0,listen


In [38]:
ground_truth_filtered.head()

,relevant_items
uid,
966200,"[1200507, 6656004, 4911665, 6060197, 2276730, ..."
828600,"[4234478, 1680781, 3388022, 5563083, 4035394, ..."
151200,"[7744857, 6666593, 2292060, 3217851, 2380340, ..."
167200,"[2449861, 2269464, 9323593, 2553666, 3087842, ..."
291400,"[5336747, 4836353, 7806672, 2759681, 826707, 4..."


### TopPop

Обучние TopPop (примитивная рекомендовалка самых популярных треков по разным критериям)

In [39]:
from src.models.counters import EventCounter, ListeningTimeCounter

listen_event_counter = EventCounter("listen_events_counter")
listen_event_counter.fit(train_df, "listen")

like_event_counter = EventCounter("like_events_counter")
like_event_counter.fit(train_df, "like")

listen_time_counter = ListeningTimeCounter("listen_time_counter")
listen_time_counter.fit(train_df)

Fitting EventCounter model for event 'listen'...
EventCounter model fitted successfully!
Fitting EventCounter model for event 'like'...
EventCounter model fitted successfully!
Fitting ListeningTimeCounter model for event 'listen'...
ListeningTimeCounter model fitted successfully!


Оценим данные примитивные модели

In [56]:
from src.metrics.evaluators import evaluate_recommenders

counter_recommenders = {
    "listen_events_counter": listen_event_counter,
    "like_events_counter": like_event_counter,
    "listen_time_counter": listen_time_counter
}

evaluate_recommenders(counter_recommenders, ground_truth_filtered)

Evaluating 'listen_events_counter'...
  precision@10 = 0.1374
  hit_rate@10  = 0.4567
  map@10       = 0.0752
  ndcg@10      = 0.1307
Evaluating 'like_events_counter'...
  precision@10 = 0.1415
  hit_rate@10  = 0.5017
  map@10       = 0.0858
  ndcg@10      = 0.1492
Evaluating 'listen_time_counter'...
  precision@10 = 0.1374
  hit_rate@10  = 0.4567
  map@10       = 0.0820
  ndcg@10      = 0.1398


,precision@10,hit_rate@10,map@10,ndcg@10
uid,,,,
listen_events_counter,0.137375,0.456678,0.075168,0.130745
like_events_counter,0.141488,0.501746,0.085809,0.149182
listen_time_counter,0.137375,0.456678,0.081955,0.139828


### ALS

In [ ]:
from src.models.matrix_factorizations import ALS

als_like = ALS("als_like", iterations=30)
als_like.fit(train_df, event_weights={"like": 1, "dislike": 0, "unlike": -1, "undislike": 0}, listen_weight=0)

als_listen = ALS("als_listen", iterations=30)
als_listen.fit(train_df, event_weights={"like": 1, "dislike": 0, "unlike": -1, "undislike": 0}, listen_weight=0.01)

als_dislike = ALS("als_dislike", iterations=30)
als_dislike.fit(train_df, event_weights={"like": 0, "dislike": -1, "unlike": 0, "undislike": 1}, listen_weight=0)

als_hybrid = ALS("als_hybrid", iterations=30)
als_hybrid.fit(train_df, event_weights={"like": 1, "dislike": -1, "unlike": -1, "undislike": 1}, listen_weight=0.01)

Preparing data for ALS model training with custom event weights...
Training ALS model...


100%|██████████| 30/30 [01:43<00:00,  3.44s/it]


ALS model trained successfully!
Preparing data for ALS model training with custom event weights...
Training ALS model...


100%|██████████| 30/30 [02:21<00:00,  4.71s/it]


ALS model trained successfully!
Preparing data for ALS model training with custom event weights...
Training ALS model...


100%|██████████| 30/30 [01:21<00:00,  2.73s/it]


ALS model trained successfully!
Preparing data for ALS model training with custom event weights...
Training ALS model...


100%|██████████| 30/30 [02:19<00:00,  4.63s/it]

ALS model trained successfully!


Оценим als-модели

In [57]:
als_recommenders = {"als_like": als_like, "als_listen": als_listen, "als_dislike": als_dislike, "als_hybrid": als_hybrid}

evaluate_recommenders(als_recommenders, ground_truth_filtered)

Evaluating 'als_like'...
  precision@10 = 0.0622
  hit_rate@10  = 0.3395
  map@10       = 0.0289
  ndcg@10      = 0.0657
Evaluating 'als_listen'...
  precision@10 = 0.0909
  hit_rate@10  = 0.4368
  map@10       = 0.0462
  ndcg@10      = 0.0962
Evaluating 'als_dislike'...
  precision@10 = 0.0074
  hit_rate@10  = 0.0643
  map@10       = 0.0030
  ndcg@10      = 0.0084
Evaluating 'als_hybrid'...
  precision@10 = 0.0910
  hit_rate@10  = 0.4370
  map@10       = 0.0462
  ndcg@10      = 0.0963


,precision@10,hit_rate@10,map@10,ndcg@10
uid,,,,
als_like,0.062178,0.339481,0.028941,0.065749
als_listen,0.090943,0.436818,0.046180,0.096234
als_dislike,0.007355,0.064273,0.003004,0.008448
als_hybrid,0.090976,0.437036,0.046196,0.096285


### Общая оценка моделей

In [59]:
all_recommenders = counter_recommenders | als_recommenders

metrcis = evaluate_recommenders(all_recommenders, ground_truth_filtered)

Evaluating 'listen_events_counter'...
  precision@10 = 0.1374
  hit_rate@10  = 0.4567
  map@10       = 0.0752
  ndcg@10      = 0.1307
Evaluating 'like_events_counter'...
  precision@10 = 0.1415
  hit_rate@10  = 0.5017
  map@10       = 0.0858
  ndcg@10      = 0.1492
Evaluating 'listen_time_counter'...
  precision@10 = 0.1374
  hit_rate@10  = 0.4567
  map@10       = 0.0820
  ndcg@10      = 0.1398
Evaluating 'als_like'...
  precision@10 = 0.0622
  hit_rate@10  = 0.3395
  map@10       = 0.0289
  ndcg@10      = 0.0657
Evaluating 'als_listen'...
  precision@10 = 0.0909
  hit_rate@10  = 0.4368
  map@10       = 0.0462
  ndcg@10      = 0.0962
Evaluating 'als_dislike'...
  precision@10 = 0.0074
  hit_rate@10  = 0.0643
  map@10       = 0.0030
  ndcg@10      = 0.0084
Evaluating 'als_hybrid'...
  precision@10 = 0.0910
  hit_rate@10  = 0.4370
  map@10       = 0.0462
  ndcg@10      = 0.0963


In [61]:
metrics = metrcis.sort_values(by="ndcg@10")
metrcis

,precision@10,hit_rate@10,map@10,ndcg@10
uid,,,,
listen_events_counter,0.137375,0.456678,0.075168,0.130745
like_events_counter,0.141488,0.501746,0.085809,0.149182
listen_time_counter,0.137375,0.456678,0.081955,0.139828
als_like,0.062178,0.339481,0.028941,0.065749
als_listen,0.090943,0.436818,0.046180,0.096234
als_dislike,0.007355,0.064273,0.003004,0.008448
als_hybrid,0.090976,0.437036,0.046196,0.096285


Смотря на результаты видим, что в потреблении пользователей есть большой bias на более популярную музыку, примитивные рекомендации самых популярных айтемов дают более выскоие метрики ndcg@10, map@10, precision@10, hit_rate@10